In [4]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report


In [5]:
data=pd.read_csv(r"C:\Users\91829\Downloads\Book1.csv")
print(data.head())

   Sl_no  Age  Cibil_score cibil  Loan_type Eligblity
0      1   22          629  Good  Not_First   Eligble
1      2   29          516  Good  Not_First   Eligble
2      3   30          652  Good  Not_First   Eligble
3      4   30          900  Good  Not_First   Eligble
4      5   27          731  Good  Not_First   Eligble


In [12]:
X = data.iloc[:, [1, 3, 4]].copy()
y = data.iloc[:, -1].copy()

# Make training data case-insensitive
X["cibil"] = X["cibil"].astype(str).str.strip().str.lower()
X["Loan_type"] = X["Loan_type"].astype(str).str.strip().str.lower()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=0
)
print(X_train.shape)

# Update these column names if they differ in your CSV
ordinal_col = ["cibil"]
nominal_col = ["Loan_type"]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "ord_cibil",
            OrdinalEncoder(
                categories=[["good", "average", "poor"]],
                handle_unknown="use_encoded_value",
                unknown_value=-1,
            ),
            ordinal_col,
        ),
        (
            "onehot_loan_type",
            OneHotEncoder(handle_unknown="ignore"),
            nominal_col,
        ),
    ],
    remainder="passthrough",
)

model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", RandomForestClassifier(n_estimators=200, random_state=0)),
    ]
)

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

def predict_eligibility(age, cibil, loan_type):
    cibil = str(cibil).strip().lower()
    loan_type = str(loan_type).strip().lower()
    new_customer = pd.DataFrame(
        [{"Age": age, "cibil": cibil, "Loan_type": loan_type}]
    )
    return model.predict(new_customer)[0]
age = int(input("Enter Age: "))
cibil = input("Enter cibil category (Good/Average/poor): ").strip().lower()
loan_type = input("Enter Loan_type (First/Not_First): ").strip().lower()
prediction = predict_eligibility(age, cibil, loan_type)
article = "an" if cibil == "average" else "a"

print(
    f"Customer of age {age}, with {article} {cibil} cibil, "
    f"as it is his/her {loan_type} loan, eligibility is: {prediction}"
)

if prediction == "Not_Eligble":
    print(f"As your cibil is {cibil}, you are not eligible for taking an iPhone on loan")

if age < 24 and cibil == "poor"or cibil == "average" and prediction == "Not_Eligble":
    print(
        f"As your age is {age} and you have {article} {cibil} cibil, "
        "you are not eligible for taking an iPhone on loan. "
        "You may try to take it without loan, or purchase it on a parent/guardian's name."
    )


(80, 3)
Accuracy: 1.0
              precision    recall  f1-score   support

     Eligble       1.00      1.00      1.00        20

    accuracy                           1.00        20
   macro avg       1.00      1.00      1.00        20
weighted avg       1.00      1.00      1.00        20

Customer of age 20, with an average cibil, as it is his/her first loan, eligibility is: Not_Eligble
As your cibil is average, you are not eligible for taking an iPhone on loan
As your age is 20 and you have an average cibil, you are not eligible for taking an iPhone on loan. You may try to take it without loan, or purchase it on a parent/guardian's name.
